# How2Sign PoseT5 -> Mixed-All Finetune
Loads the latest How2Sign pretrain output from Kaggle, copies it into a fresh working checkpoint directory, finetunes on the mixed-all Thai dataset, then exports and verifies a candidate runtime artifact.

In [ ]:
import csv, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
PRETRAIN_SLUG = os.environ.get('TSL_PRETRAIN_SLUG', 'thai-sign-how2sign-pretrain-output')

def find_input_mount(slug: str) -> str:
    candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / 'datasets' / 'orbitorls' / slug,
        INPUT_ROOT / 'kernels' / 'orbitorls' / slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return str(candidate)
    return ''

def materialize_manifest_dataset(dataset_root: str, working_slug: str) -> str:
    dataset_path = Path(dataset_root)
    manifest_path = dataset_path / 'manifest.csv'
    if not manifest_path.is_file():
        raise FileNotFoundError({'dataset_root': dataset_root, 'missing': 'manifest.csv'})

    def manifest_feature_probe(root: Path) -> dict:
        matched = 0
        total = 0
        samples = []
        with manifest_path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rel_path = str(row.get('npy_path', '')).replace('\\', '/')
                if not rel_path:
                    continue
                total += 1
                candidate = root / rel_path
                if candidate.is_file():
                    matched += 1
                    if matched >= 3:
                        break
                elif len(samples) < 3:
                    samples.append(rel_path)
        return {'matched': matched, 'total': total, 'missing_samples': samples}

    direct_features = dataset_path / 'features'
    direct_probe = manifest_feature_probe(dataset_path)
    if direct_probe['matched'] > 0:
        return str(dataset_path)

    nested_features = direct_features / 'features'
    if nested_features.is_dir() and any(nested_features.rglob('*.npy')):
        data_runtime = Path('/kaggle/working/datasets') / working_slug
        shutil.rmtree(data_runtime, ignore_errors=True)
        data_runtime.mkdir(parents=True, exist_ok=True)
        for name in ['manifest.csv', 'manifest_quality.json']:
            src = dataset_path / name
            if src.is_file():
                shutil.copy2(src, data_runtime / name)
        shutil.copytree(nested_features, data_runtime / 'features', dirs_exist_ok=True)
        repaired_probe = manifest_feature_probe(data_runtime)
        if repaired_probe['matched'] > 0:
            return str(data_runtime)

    archive_candidates = [dataset_path / 'features.zip']
    archive_candidates.extend(sorted(dataset_path.rglob('features.zip')))
    archive_path = next((path for path in archive_candidates if path.is_file()), None)
    if archive_path is None:
        raise FileNotFoundError({
            'dataset_root': dataset_root,
            'missing': 'features or features.zip',
            'visible': sorted(path.name for path in dataset_path.iterdir()),
        })

    data_runtime = Path('/kaggle/working/datasets') / working_slug
    shutil.rmtree(data_runtime, ignore_errors=True)
    data_runtime.mkdir(parents=True, exist_ok=True)
    for name in ['manifest.csv', 'manifest_quality.json']:
        src = dataset_path / name
        if src.is_file():
            shutil.copy2(src, data_runtime / name)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(data_runtime)

    materialized_features = data_runtime / 'features'
    extracted_probe = manifest_feature_probe(data_runtime)
    if not materialized_features.is_dir() or extracted_probe['matched'] == 0:
        raise FileNotFoundError({
            'dataset_root': dataset_root,
            'archive_path': str(archive_path),
            'materialized_root': str(data_runtime),
            'missing': 'materialized manifest feature files',
            'probe': extracted_probe,
        })
    return str(data_runtime)

def query_gpu_info() -> tuple[str, float, str]:
    result = subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,compute_cap,memory.total',
        '--format=csv,noheader',
    ], text=True).strip().splitlines()[0]
    name, capability, memory_total = [part.strip() for part in result.split(',', 2)]
    return name, float(capability), memory_total

def ensure_compatible_torch(gpu_capability: float) -> dict:
    selected = None
    if gpu_capability < 7.0:
        compatible_candidates = [
            ('2.2.0', '0.17.0', '2.2.0'),
            ('2.2.1', '0.17.1', '2.2.1'),
            ('2.2.2', '0.17.2', '2.2.2'),
            ('2.3.0', '0.18.0', '2.3.0'),
            ('2.3.1', '0.18.1', '2.3.1'),
            ('2.4.0', '0.19.0', '2.4.0'),
            ('2.4.1', '0.19.1', '2.4.1'),
            ('2.5.0', '0.20.0', '2.5.0'),
            ('2.5.1', '0.20.1', '2.5.1'),
        ]
        for torch_version, torchvision_version, torchaudio_version in compatible_candidates:
            subprocess.check_call([
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                f'torch=={torch_version}',
                f'torchvision=={torchvision_version}',
                f'torchaudio=={torchaudio_version}',
                '--index-url',
                'https://download.pytorch.org/whl/cu121',
            ])
            arch_probe = subprocess.check_output([
                sys.executable,
                '-c',
                'import json, torch; print(json.dumps(sorted(torch.cuda.get_arch_list())))',
            ], text=True).strip()
            print({'candidate_torch': torch_version, 'arch_probe': arch_probe})
            if 'sm_60' in arch_probe:
                selected = {
                    'torch': torch_version,
                    'torchvision': torchvision_version,
                    'torchaudio': torchaudio_version,
                    'wheel_index': 'cu121',
                }
                break
        if selected is None:
            raise RuntimeError('No available cu121 torch wheel in Kaggle exposed sm_60 support for P100.')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy<2'])
    return selected or {'torch': 'default', 'wheel_index': 'default'}

CODE_DATASET = find_input_mount('thai-sign-code')
DATA = find_input_mount('thai-sign-mixed-all-v6-archived')
MT5_LOCAL = find_input_mount('thai-sign-mt5small')
PRETRAIN_ROOT = find_input_mount(PRETRAIN_SLUG)
CKPT_DIR = '/kaggle/working/pose_t5_mixed_all_from_how2sign'
CODE = '/tmp/thai_sign_code'
DATA_RUNTIME = materialize_manifest_dataset(DATA, 'thai-sign-mixed-all-v6-archived')

missing = {
    'thai-sign-code': CODE_DATASET,
    'thai-sign-mixed-all-v6-archived': DATA,
    'thai-sign-mt5small': MT5_LOCAL,
    PRETRAIN_SLUG: PRETRAIN_ROOT,
}
missing = [name for name, path in missing.items() if not path]
if missing:
    raise FileNotFoundError({'missing_inputs': missing, 'input_root': str(INPUT_ROOT), 'visible_inputs': sorted(p.name for p in INPUT_ROOT.iterdir())})

gpu_name, gpu_capability, gpu_memory = query_gpu_info()
selected_torch = ensure_compatible_torch(gpu_capability)
for package in ['transformers>=4.40', 'sentencepiece>=0.2.0', 'sacrebleu>=2.4.0', 'portalocker>=2.0', 'pandas>=2.0']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)
repo_bundle = os.path.join(CODE_DATASET, 'repo_bundle.zip')
if os.path.isfile(repo_bundle):
    with zipfile.ZipFile(repo_bundle) as archive:
        archive.extractall(CODE)
else:
    for name in ['config.py', 'requirements.txt', 'README.md']:
        src = os.path.join(CODE_DATASET, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(CODE, name))
    for name in ['src', 'scripts']:
        src_dir = os.path.join(CODE_DATASET, name)
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join(CODE, name), dirs_exist_ok=True)
shutil.rmtree(CKPT_DIR, ignore_errors=True)
os.makedirs(CKPT_DIR, exist_ok=True)
for src in Path(PRETRAIN_ROOT).iterdir():
    dst = Path(CKPT_DIR) / src.name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
if not os.path.isfile(os.path.join(CKPT_DIR, 'best_model_state.pt')):
    raise FileNotFoundError({'pretrain_root': PRETRAIN_ROOT, 'expected': 'best_model_state.pt', 'found': sorted(p.name for p in Path(PRETRAIN_ROOT).iterdir())})

print({
    'gpu_name': gpu_name,
    'gpu_capability': gpu_capability,
    'gpu_memory': gpu_memory,
    'selected_torch': selected_torch,
    'code_dataset': CODE_DATASET,
    'data': DATA,
    'data_runtime': DATA_RUNTIME,
    'mt5_local': MT5_LOCAL,
    'pretrain_root': PRETRAIN_ROOT,
    'seed_files': sorted(os.listdir(CKPT_DIR)),
})


In [ ]:
import os, subprocess, sys

base_model = MT5_LOCAL if os.path.isdir(MT5_LOCAL) else 'google/mt5-small'
train_script_candidates = [
    os.path.join(CODE, 'scripts', 'kaggle_train_pose_t5.py'),
    os.path.join(CODE, 'kaggle_train_pose_t5.py'),
]
refresh_script_candidates = [
    os.path.join(CODE, 'scripts', 'refresh_pose_t5_verified.py'),
    os.path.join(CODE, 'refresh_pose_t5_verified.py'),
]
search_script_candidates = [
    os.path.join(CODE, 'scripts', 'search_pose_t5_decoding.py'),
    os.path.join(CODE, 'search_pose_t5_decoding.py'),
]
train_script = next((path for path in train_script_candidates if os.path.isfile(path)), None)
refresh_script = next((path for path in refresh_script_candidates if os.path.isfile(path)), None)
search_script = next((path for path in search_script_candidates if os.path.isfile(path)), None)
if train_script is None or refresh_script is None or search_script is None:
    raise FileNotFoundError({'train_script': train_script, 'refresh_script': refresh_script, 'search_script': search_script, 'code_contents': sorted(os.listdir(CODE))})

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([os.path.join(CODE, 'src'), CODE, env.get('PYTHONPATH', '')])
if os.path.isdir(MT5_LOCAL):
    env['TRANSFORMERS_OFFLINE'] = '1'
    env['HF_HUB_OFFLINE'] = '1'

train_cmd = [
    sys.executable,
    train_script,
    '--data-roots', DATA_RUNTIME,
    '--out-dir', CKPT_DIR,
    '--base-model', base_model,
    '--epochs', '120',
    '--batch-size', '4',
    '--grad-accum', '4',
    '--amp', 'auto',
    '--max-runtime-min', '690',
    '--eval-steps', '20',
    '--checkpoint-steps', '5000',
    '--resume', 'best_state',
    '--lr', '1e-5',
    '--dropout', '0.1',
    '--balance-sources', 'auto',
    '--preload-train-features', 'false',
    '--required-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--manifest-quality-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--fail-on-manifest-quality', 'true',
    '--allow-noop-resume', 'false',
    '--split-policy', 'manifest',
    '--early-stopping-metric', 'val_chrf',
    '--early-stopping-patience', '10',
    '--reset-progress-history',
    '--evaluate-after-train', 'true',
    '--eval-data-roots', DATA_RUNTIME,
    '--eval-report-data-roots', 'data/mixed_all_train_v6',
    '--eval-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--eval-split-policy', 'manifest',
    '--eval-device', 'cpu',
    '--eval-val-subset-size', '60',
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True, env=env)

candidate_export_dir = os.path.join(CKPT_DIR, 'candidate_export')
verified_export_dir = os.path.join(CKPT_DIR, 'verified_export')
refresh_cmd = [
    sys.executable,
    refresh_script,
    '--train-dir', CKPT_DIR,
    '--candidate-export-dir', candidate_export_dir,
    '--verified-export-dir', verified_export_dir,
    '--candidate-eval-json', os.path.join(CKPT_DIR, 'candidate_eval.json'),
    '--verified-eval-json', os.path.join(CKPT_DIR, 'verified_eval.json'),
    '--candidate-samples-json', os.path.join(CKPT_DIR, 'candidate_samples.json'),
    '--verified-samples-json', os.path.join(CKPT_DIR, 'verified_samples.json'),
    '--checkpoint', 'best_state',
    '--data-roots', DATA_RUNTIME,
    '--device', 'cpu',
    '--seed', '42',
    '--val-subset-size', '60',
    '--base-model', base_model,
    '--input-dim', '312',
    '--num-encoder-layers', '2',
    '--dropout', '0.1',
    '--downsample-factor', '4',
    '--min-source-examples', '5',
    '--min-source-chrf', '20',
    '--min-source-exact-match-pct', '5',
]
print('Running:', ' '.join(refresh_cmd))
subprocess.run(refresh_cmd, check=True, env=env)

search_cmd = [
    sys.executable,
    search_script,
    '--export-dir', verified_export_dir,
    '--data-roots', DATA_RUNTIME,
    '--report-data-roots', 'data/mixed_all_train_v6',
    '--required-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--manifest-quality-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--eval-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--split-policy', 'manifest',
    '--val-subset-size', '60',
    '--device', 'cpu',
    '--max-trials', '32',
]
print('Running:', ' '.join(search_cmd))
subprocess.run(search_cmd, check=True, env=env)


In [ ]:
import json, os
from pathlib import Path

files = sorted(os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else []
print(files)
run_status_path = Path(CKPT_DIR) / 'run_status.json'
verified_eval_path = Path(CKPT_DIR) / 'verified_eval.json'
if run_status_path.is_file():
    print('run_status.json')
    print(json.loads(run_status_path.read_text(encoding='utf-8')))
if verified_eval_path.is_file():
    print('verified_eval.json')
    print(json.loads(verified_eval_path.read_text(encoding='utf-8')))
